In [28]:
from collections import Counter
from pathlib import Path
import re

import pandas as pd

In [29]:
# Find all CSV files in the data folder

DATA_PATH = Path("../data/Translated")
csv_files = sorted(DATA_PATH.glob("*.csv"))

print(f"Found {len(csv_files)} files:")
for f in csv_files:
    print(f.name)

Found 17 files:
Speeches_ALDE_Cleaned_Translated.csv
Speeches_CZ_Cleaned_Translated.csv
Speeches_DE_Cleaned_Translated.csv
Speeches_ECB_Cleaned_Translated.csv
Speeches_ECR_Cleaned_Translated.csv
Speeches_EC_Cleaned_Translated.csv
Speeches_EP_Cleaned_Translated.csv
Speeches_EUCouncil_Cleaned_Translated.csv
Speeches_FR_Cleaned_Translated.csv
Speeches_GR_Cleaned_Translated.csv
Speeches_IMF_Cleaned_Translated.csv
Speeches_IT_Cleaned_Translated.csv
Speeches_NL_Cleaned_Translated.csv
Speeches_PL_Cleaned_Translated.csv
Speeches_PO_Cleaned_Translated.csv
Speeches_SP_Cleaned_Translated.csv
Speeches_UK_Cleaned_Translated.csv


In [30]:
# Load all CSV files into DataFrame

ALL_COLUMNS = ["title", "date", "location", "speaker", "word_count", "raw", "raw_text", "original", "lang"]
USE_COLUMNS = ["title", "date", "location", "speaker", "raw"]

def clean_sentence(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def split_sentences(text: str) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [clean_sentence(s) for s in sentences if s.strip()]

def load_speech_csv(path: Path) -> pd.DataFrame:
    nation = path.stem.split("_")[1]
    df = pd.read_csv(path, header=None, names=ALL_COLUMNS, usecols=USE_COLUMNS)
    df["nation"] = nation
    df["sentences"] = df["raw"].apply(split_sentences)
    return df

corpus = pd.concat([load_speech_csv(f) for f in csv_files], ignore_index=True)
print(f"Loaded {len(corpus)} speeches:")
corpus.head()

Loaded 6106 speeches:


,title,date,location,speaker,raw,nation,sentences
0,Rede von Guy Verhofstadt anläßlich des FDP Eur...,23-06-2013,European Parliament,Guy Verhofstadt,"Love Nadja, dear friends,Dear Sir or Madam,rea...",ALDE,[love nadja dear friends dear sir or madam rea...
1,Speech by Guy Verhofstadt - Intervention in pl...,05-02-2013,European Parliament,Guy Verhofstadt,Intervention Guy VerhofstadtNovember 21 2012Vi...,ALDE,[intervention guy verhofstadtnovember 21 2012v...
2,Speech by Guy Verhofstadt on the future of Europe,09-05-2012,European Parliament,Guy Verhofstadt,"Dear President, Dear colleagues,Let me paraphr...",ALDE,[dear president dear colleagues let me paraphr...
3,Speech by Guy Verhofstadt on PPV,12-03-2012,European Parliament,Guy Verhofstadt,"My group, President, for the initiative of Mr ...",ALDE,[my group president for the initiative of mr w...
4,Le Grand Direct de Guy Verhofstadt,07-03-2012,European Parliament,Guy Verhofstadt,Guy Verhofstadt Live dupuis RTL-TVI answering ...,ALDE,[guy verhofstadt live dupuis rtl tvi answering...


In [31]:
corpus.groupby("nation").size().sort_values(ascending=False).rename("speech_count")

nation
FR           1451
EP           1064
SP            996
IT            804
DE            580
GR            390
NL            260
CZ            234
EC            149
PO            133
ECB            18
EUCouncil      16
ALDE            6
PL              4
ECR             1
Name: speech_count, dtype: int64